# 10 - Deploy Serving, BI, App, and Agent Artifacts

Executes demo-owned Warehouse SQL and Eventhouse KQL, deploys the TMDL semantic model, converts the portable PBIR path binding to a runtime semantic-model connection, and configures/publishes the native Data Agent through its staging APIs. Fabric app capability is checked explicitly; unsupported item types are recorded and never reported as success.

In [ ]:
# PARAMETERS - supply endpoint values at runtime; never commit them.
workspace_id = ''
artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
warehouse_sql_endpoint = ''
warehouse_database_name = 'AirportOpsWarehouse'
kql_query_uri = ''
kql_database_name = 'AirportOpsRealtime'
kql_token_audience = 'kusto'
environment_name = 'demo'
dry_run = True
execute_warehouse_sql = True
execute_eventhouse_kql = True
deploy_bi_definitions = True
deploy_conditional_artifacts = False
strict_mode = True
poll_timeout_seconds = 600
git_commit = ''
deployment_manifest_output = '/lakehouse/default/Files/airport-ops-mvp/deployment/runtime-platform-manifest.json'

import base64
import json
import re
import struct
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

API_BASE = 'https://api.fabric.microsoft.com/v1'


In [ ]:
ROOT = Path(artifact_root)
DEPLOYMENT_RUN_ID = 'RUN-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTS = []

if not dry_run:
    assert re.fullmatch(r'[0-9a-fA-F-]{36}', workspace_id), 'A runtime Fabric workspace GUID is required'
forbidden_secret_markers = ['accountkey=', 'sharedaccesskey=', 'pass' + 'word=', 'bearer ']
assert not any(term in (warehouse_sql_endpoint + kql_query_uri).lower() for term in forbidden_secret_markers)


def record(name, artifact_type, status, detail='', item_id='', request_id=''):
    row = {
        'deployment_run_id': DEPLOYMENT_RUN_ID,
        'environment_name': environment_name,
        'artifact_name': name,
        'artifact_type': artifact_type,
        'deployment_method': 'Notebook SQL/KQL or Fabric REST item definition',
        'deployment_status': status,
        'dependency_status': 'SATISFIED' if status not in {'FAILED', 'SKIPPED_PREREQUISITE'} else 'UNSATISFIED',
        'validation_status': 'PENDING' if status == 'SUCCEEDED' else status,
        'unsupported_manual_status': 'MANUAL_OR_UNSUPPORTED' if status in {'SKIPPED_PREREQUISITE', 'SKIPPED_UNSUPPORTED'} else '',
        'error_details': detail[:4000] if status == 'FAILED' else '',
        'status_detail': detail[:4000],
        'item_id': item_id or '',
        'request_id': request_id or '',
        'observed_at': datetime.now(timezone.utc),
        'git_commit': git_commit or '',
        'is_synthetic': True,
    }
    RESULTS.append(row)
    print(status, artifact_type, name, detail)
    return row


In [ ]:
class FabricApiError(RuntimeError):
    def __init__(self, status_code, message, request_id=''):
        super().__init__(message)
        self.status_code = status_code
        self.request_id = request_id


def auth_headers(audience='pbi'):
    return {'Authorization': 'Bearer ' + notebookutils.credentials.getToken(audience), 'Content-Type': 'application/json'}


def request_id(response):
    return response.headers.get('x-ms-request-id') or response.headers.get('requestId') or ''


def fabric_request(method, path, payload=None, accepted=(200, 201, 202)):
    response = None
    for attempt in range(6):
        response = requests.request(method, API_BASE + path, headers=auth_headers(), json=payload, timeout=90)
        if response.status_code in accepted:
            break
        if response.status_code in {429, 500, 502, 503, 504}:
            time.sleep(min(2 ** attempt, 30))
            continue
        raise FabricApiError(response.status_code, response.text[:4000], request_id(response))
    if response is None or response.status_code not in accepted:
        raise FabricApiError(response.status_code if response else 0, response.text[:4000] if response else 'No response', request_id(response) if response else '')
    if response.status_code == 202 and response.headers.get('Location'):
        operation_url = response.headers['Location']
        started = time.time()
        while time.time() - started < poll_timeout_seconds:
            operation = requests.get(operation_url, headers=auth_headers(), timeout=90)
            body = operation.json() if operation.content else {}
            status = str(body.get('status', '')).lower()
            if status in {'succeeded', 'completed'}:
                return operation, body
            if status in {'failed', 'cancelled'}:
                raise FabricApiError(operation.status_code, json.dumps(body)[:4000], request_id(operation))
            time.sleep(int(operation.headers.get('Retry-After', '3')))
        raise TimeoutError('Fabric operation timed out')
    return response, response.json() if response.content else {}


def list_items(item_type):
    _, body = fabric_request('GET', '/workspaces/' + workspace_id + '/items?type=' + item_type, accepted=(200,))
    return body.get('value', [])


def ensure_item(display_name, item_type):
    if dry_run:
        return {'id': 'DRYRUN-' + item_type, 'displayName': display_name, 'type': item_type}
    matches = [item for item in list_items(item_type) if item.get('displayName') == display_name]
    if len(matches) > 1:
        raise RuntimeError('Multiple items named ' + display_name)
    if matches:
        return matches[0]
    response, _ = fabric_request('POST', '/workspaces/' + workspace_id + '/items', {
        'displayName': display_name,
        'type': item_type,
        'description': 'Synthetic airport operations demo; advisory only',
    })
    matches = [item for item in list_items(item_type) if item.get('displayName') == display_name]
    if not matches:
        raise RuntimeError('Created item could not be resolved: ' + display_name)
    record(display_name, item_type, 'SUCCEEDED', 'Created item', matches[0]['id'], request_id(response))
    return matches[0]


def definition_parts(definition_root, replacements=None):
    root = Path(definition_root)
    if not root.exists():
        raise FileNotFoundError(str(root))
    parts = []
    replacements = replacements or {}
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.name == '.platform' or path.suffix == '.py':
            continue
        data = path.read_bytes()
        if path.suffix.lower() in {'.json', '.tmdl', '.pbism', '.pbir', '.md', '.yaml', '.kql', '.sql'}:
            text = data.decode('utf-8')
            for token, value in replacements.items():
                text = text.replace(token, value)
            if '${WAREHOUSE_' in text:
                raise ValueError('Unresolved Warehouse parameter in ' + str(path))
            data = text.encode('utf-8')
        parts.append({
            'path': path.relative_to(root).as_posix(),
            'payload': base64.b64encode(data).decode('ascii'),
            'payloadType': 'InlineBase64',
        })
    if not parts:
        raise ValueError('No definition parts found under ' + str(root))
    return parts


def deploy_definition(display_name, item_type, definition_root, replacements=None, conditional=False):
    if dry_run:
        part_count = len(definition_parts(definition_root, replacements))
        record(display_name, item_type, 'DRY_RUN', 'Would deploy ' + str(part_count) + ' definition parts')
        return
    try:
        item = ensure_item(display_name, item_type)
        response, _ = fabric_request(
            'POST', '/workspaces/' + workspace_id + '/items/' + item['id'] + '/updateDefinition',
            {'definition': {'parts': definition_parts(definition_root, replacements)}},
        )
        record(display_name, item_type, 'SUCCEEDED', 'Definition deployed', item['id'], request_id(response))
    except FabricApiError as exc:
        if conditional and exc.status_code in {400, 404, 405, 409, 422}:
            record(display_name, item_type, 'SKIPPED_UNSUPPORTED', str(exc), request_id=exc.request_id)
            return
        record(display_name, item_type, 'FAILED', str(exc), request_id=exc.request_id)
        if strict_mode:
            raise

In [ ]:
# Definition-backed items are created atomically with their initial definition, then updated on reruns.
def deploy_definition(display_name, item_type, definition_root, replacements=None, conditional=False, parts_override=None):
    try:
        parts = parts_override or definition_parts(definition_root, replacements)
        if dry_run:
            record(display_name, item_type, 'DRY_RUN', 'Would create or update ' + str(len(parts)) + ' definition parts')
            return {'id': 'DRYRUN-' + item_type, 'displayName': display_name, 'type': item_type}

        matches = [item for item in list_items(item_type) if item.get('displayName') == display_name]
        if len(matches) > 1:
            raise RuntimeError('Multiple ' + item_type + ' items named ' + display_name)

        if matches:
            item = matches[0]
            response, _ = fabric_request(
                'POST', '/workspaces/' + workspace_id + '/items/' + item['id'] + '/updateDefinition',
                {'definition': {'parts': parts}},
            )
            record(display_name, item_type, 'SUCCEEDED', 'Definition updated', item['id'], request_id(response))
            return item

        response, _ = fabric_request(
            'POST', '/workspaces/' + workspace_id + '/items',
            {
                'displayName': display_name,
                'type': item_type,
                'description': 'Synthetic airport operations demo; advisory only',
                'definition': {'parts': parts},
            },
        )
        created = [item for item in list_items(item_type) if item.get('displayName') == display_name]
        if len(created) != 1:
            raise RuntimeError('Created definition item could not be uniquely resolved: ' + display_name)
        record(display_name, item_type, 'SUCCEEDED', 'Created item', created[0]['id'], request_id(response))
        return created[0]
    except FabricApiError as exc:
        if conditional and exc.status_code in {400, 404, 405, 409, 422}:
            record(display_name, item_type, 'SKIPPED_UNSUPPORTED', str(exc), request_id=exc.request_id)
            return None
        record(display_name, item_type, 'FAILED', str(exc), request_id=exc.request_id)
        if strict_mode:
            raise
    except Exception as exc:
        record(display_name, item_type, 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
        if strict_mode:
            raise
    return None


def report_definition_parts(definition_root, semantic_model_id):
    parts = definition_parts(definition_root)
    if dry_run:
        return parts
    _, workspace = fabric_request('GET', '/workspaces/' + workspace_id, accepted=(200,))
    connection_string = (
        'Data Source=powerbi://api.powerbi.com/v1.0/myorg/' + workspace['displayName'] +
        ';Initial Catalog=AirportOpsSharedModel;semanticModelId=' + semantic_model_id +
        ';Integrated Security=ClaimsToken'
    )
    for part in parts:
        if part['path'] == 'definition.pbir':
            definition = json.loads(base64.b64decode(part['payload']).decode('utf-8'))
            definition['datasetReference'] = {'byConnection': {'connectionString': connection_string}}
            part['payload'] = base64.b64encode(json.dumps(definition, separators=(',', ':')).encode('utf-8')).decode('ascii')
            break
    else:
        raise ValueError('Report definition.pbir part was not found')
    return parts


def data_agent_elements(data_agent_id, datasource_id, root_id):
    path = (
        '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id +
        '/staging/datasources/' + datasource_id + '/elements?rootId=' + requests.utils.quote(root_id, safe='')
    )
    _, body = fabric_request('GET', path, accepted=(200,))
    return body.get('value', [])


def deploy_data_agent():
    display_name = 'AirportOpsDataAgent'
    if dry_run:
        definition = json.loads((ROOT / 'data-agent' / 'definition.json').read_text(encoding='utf-8'))
        record(display_name, 'DataAgent', 'DRY_RUN', 'Would publish one Warehouse source with ' + str(len(definition['approved_warehouse_sources'])) + ' curated views')
        return
    try:
        matches = [item for item in list_items('DataAgent') if item.get('displayName') == display_name]
        if len(matches) > 1:
            raise RuntimeError('Multiple DataAgent items named ' + display_name)
        if matches:
            item = matches[0]
        else:
            response, _ = fabric_request(
                'POST', '/workspaces/' + workspace_id + '/items',
                {'displayName': display_name, 'type': 'DataAgent', 'description': 'Synthetic airport operations read-only governed data agent; advisory only'},
                accepted=(201, 202),
            )
            created = [candidate for candidate in list_items('DataAgent') if candidate.get('displayName') == display_name]
            if len(created) != 1:
                raise RuntimeError('Created DataAgent could not be uniquely resolved')
            item = created[0]
            record(display_name, 'DataAgent', 'SUCCEEDED', 'Created native item', item['id'], request_id(response))

        data_agent_id = item['id']
        instructions = (ROOT / 'ontology' / 'data-agent-instructions.md').read_text(encoding='utf-8')
        fabric_request(
            'PATCH', '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/settings',
            {'aiInstructions': instructions}, accepted=(200,),
        )

        warehouses = [candidate for candidate in list_items('Warehouse') if candidate.get('displayName') == warehouse_database_name]
        if len(warehouses) != 1:
            raise RuntimeError('Warehouse item could not be uniquely resolved: ' + warehouse_database_name)
        warehouse_id = warehouses[0]['id']
        datasource_path = '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/datasources'
        _, datasource_body = fabric_request('GET', datasource_path, accepted=(200,))
        datasources = datasource_body.get('value', [])
        datasource = next((source for source in datasources if source.get('itemReference', {}).get('itemId') == warehouse_id), None)
        if datasource is None:
            fabric_request(
                'POST', datasource_path,
                {'type': 'FabricItem', 'itemReference': {'referenceType': 'ById', 'itemId': warehouse_id, 'workspaceId': workspace_id}},
                accepted=(201, 202),
            )
            _, datasource_body = fabric_request('GET', datasource_path, accepted=(200,))
            datasource = next((source for source in datasource_body.get('value', []) if source.get('itemReference', {}).get('itemId') == warehouse_id), None)
        if datasource is None:
            raise RuntimeError('Warehouse DataAgent datasource was not created')
        datasource_id = datasource['id']

        schemas = data_agent_elements(data_agent_id, datasource_id, 'U2NoZW1hcw==')
        ops_schema = next((element for element in schemas if element.get('displayName') == 'ops'), None)
        if ops_schema is None:
            raise RuntimeError('Warehouse ops schema is unavailable to DataAgent')
        branches = data_agent_elements(data_agent_id, datasource_id, ops_schema['id'])
        views_branch = next((element for element in branches if element.get('displayName') == 'Views'), None)
        if views_branch is None:
            raise RuntimeError('Warehouse ops Views branch is unavailable to DataAgent')
        views = data_agent_elements(data_agent_id, datasource_id, views_branch['id'])

        definition = json.loads((ROOT / 'data-agent' / 'definition.json').read_text(encoding='utf-8'))
        allowed = {source.replace('ops.', '') for source in definition['approved_warehouse_sources']}
        available = {view['displayName'] for view in views}
        missing = sorted(allowed - available)
        if missing:
            raise RuntimeError('Allowlisted Warehouse views are unavailable: ' + ', '.join(missing))
        element_path = '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/datasources/' + datasource_id + '/elements'
        for view in views:
            should_select = view['displayName'] in allowed
            if bool(view.get('isSelected')) != should_select:
                fabric_request(
                    'PATCH', element_path + '?id=' + requests.utils.quote(view['id'], safe=''),
                    {'isSelected': should_select}, accepted=(200,),
                )

        selected = {view['displayName'] for view in data_agent_elements(data_agent_id, datasource_id, views_branch['id']) if view.get('isSelected')}
        if selected != allowed:
            raise RuntimeError('Published DataAgent source selection does not match the allowlist')
        response, _ = fabric_request(
            'POST', '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/publish',
            {'publishedDescription': 'Governed read-only airport operations agent; synthetic data; advisory only'},
            accepted=(200, 202),
        )
        record(display_name, 'DataAgent', 'SUCCEEDED', 'Published native item with one Warehouse source and ' + str(len(selected)) + ' curated views', item['id'], request_id(response))
    except Exception as exc:
        record(display_name, 'DataAgent', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
        if strict_mode:
            raise

In [ ]:
# Extend native Data Agent publication with the curated KQL source after Warehouse staging succeeds.
deploy_warehouse_data_agent = deploy_data_agent


def deploy_data_agent():
    deploy_warehouse_data_agent()
    if dry_run:
        definition = json.loads((ROOT / 'data-agent' / 'definition.json').read_text(encoding='utf-8'))
        record('AirportOpsDataAgent KQL grounding', 'DataAgent', 'DRY_RUN', 'Would publish ' + str(len(definition['approved_kql_functions'])) + ' curated KQL functions/views and no raw tables')
        return

    try:
        agents = [item for item in list_items('DataAgent') if item.get('displayName') == 'AirportOpsDataAgent']
        databases = [item for item in list_items('KQLDatabase') if item.get('displayName') == kql_database_name]
        if len(agents) != 1 or len(databases) != 1:
            raise RuntimeError('DataAgent or KQLDatabase could not be uniquely resolved')
        data_agent_id = agents[0]['id']
        kql_database_id = databases[0]['id']
        datasource_path = '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/datasources'
        _, datasource_body = fabric_request('GET', datasource_path, accepted=(200,))
        datasource = next((source for source in datasource_body.get('value', []) if source.get('itemReference', {}).get('itemId') == kql_database_id), None)
        if datasource is None:
            fabric_request(
                'POST', datasource_path,
                {'type': 'FabricItem', 'itemReference': {'referenceType': 'ById', 'itemId': kql_database_id, 'workspaceId': workspace_id}},
                accepted=(201, 202),
            )
            _, datasource_body = fabric_request('GET', datasource_path, accepted=(200,))
            datasource = next((source for source in datasource_body.get('value', []) if source.get('itemReference', {}).get('itemId') == kql_database_id), None)
        if datasource is None:
            raise RuntimeError('KQLDatabase DataAgent datasource was not created')
        datasource_id = datasource['id']
        fabric_request('PATCH', datasource_path + '/' + datasource_id, {}, accepted=(200,))

        roots = data_agent_elements(data_agent_id, datasource_id, '')
        branches = {element['displayName']: element for element in roots}
        if not {'Functions', 'Materialized Views', 'Tables'}.issubset(branches):
            raise RuntimeError('KQL DataAgent datasource branches are incomplete')
        query_elements = (
            data_agent_elements(data_agent_id, datasource_id, branches['Functions']['id']) +
            data_agent_elements(data_agent_id, datasource_id, branches['Materialized Views']['id'])
        )
        definition = json.loads((ROOT / 'data-agent' / 'definition.json').read_text(encoding='utf-8'))
        allowed = set(definition['approved_kql_functions'])
        available = {element['displayName'] for element in query_elements}
        missing = sorted(allowed - available)
        if missing:
            raise RuntimeError('Allowlisted KQL functions/views are unavailable: ' + ', '.join(missing))
        element_path = '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/datasources/' + datasource_id + '/elements'
        for element in query_elements:
            should_select = element['displayName'] in allowed
            if bool(element.get('isSelected')) != should_select:
                fabric_request(
                    'PATCH', element_path + '?id=' + requests.utils.quote(element['id'], safe=''),
                    {'isSelected': should_select}, accepted=(200,),
                )
        selected = {
            element['displayName']
            for element in (
                data_agent_elements(data_agent_id, datasource_id, branches['Functions']['id']) +
                data_agent_elements(data_agent_id, datasource_id, branches['Materialized Views']['id'])
            )
            if element.get('isSelected')
        }
        selected_tables = [
            element for element in data_agent_elements(data_agent_id, datasource_id, branches['Tables']['id'])
            if element.get('isSelected')
        ]
        if selected != allowed or selected_tables:
            raise RuntimeError('Published DataAgent KQL selection does not match the allowlist or includes raw tables')
        response, _ = fabric_request(
            'POST', '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/publish',
            {'publishedDescription': 'Governed read-only airport operations agent; curated Warehouse and KQL sources; synthetic data; advisory only'},
            accepted=(200, 202),
        )
        record('AirportOpsDataAgent KQL grounding', 'DataAgent', 'SUCCEEDED', 'Published ' + str(len(selected)) + ' curated KQL functions/views and no raw tables', data_agent_id, request_id(response))
    except Exception as exc:
        record('AirportOpsDataAgent KQL grounding', 'DataAgent', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
        if strict_mode:
            raise

In [ ]:
# Enforce the final two-source boundary after Warehouse and KQL staging are configured.
deploy_curated_data_agent = deploy_data_agent


def deploy_data_agent():
    deploy_curated_data_agent()
    if dry_run:
        record('AirportOpsDataAgent source boundary', 'DataAgent', 'DRY_RUN', 'Would enforce exactly one Warehouse and one KQLDatabase source')
        return

    try:
        agents = [item for item in list_items('DataAgent') if item.get('displayName') == 'AirportOpsDataAgent']
        warehouses = [item for item in list_items('Warehouse') if item.get('displayName') == warehouse_database_name]
        databases = [item for item in list_items('KQLDatabase') if item.get('displayName') == kql_database_name]
        if len(agents) != 1 or len(warehouses) != 1 or len(databases) != 1:
            raise RuntimeError('DataAgent, Warehouse, or KQLDatabase could not be uniquely resolved')
        data_agent_id = agents[0]['id']
        allowed_item_ids = {warehouses[0]['id'], databases[0]['id']}
        staging_path = '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/datasources'
        _, staging_body = fabric_request('GET', staging_path, accepted=(200,))
        staging_sources = staging_body.get('value', [])
        for datasource in staging_sources:
            if datasource.get('itemReference', {}).get('itemId') not in allowed_item_ids:
                fabric_request('DELETE', staging_path + '/' + datasource['id'], accepted=(200, 204))
        response, _ = fabric_request(
            'POST', '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/staging/publish',
            {'publishedDescription': 'Governed read-only airport operations agent; exactly one Warehouse and one KQL source; synthetic data; advisory only'},
            accepted=(200, 202),
        )
        _, published_body = fabric_request(
            'GET', '/workspaces/' + workspace_id + '/dataAgents/' + data_agent_id + '/datasources',
            accepted=(200,),
        )
        published_ids = {source.get('itemReference', {}).get('itemId') for source in published_body.get('value', [])}
        if published_ids != allowed_item_ids:
            raise RuntimeError('Published DataAgent sources do not match the Warehouse/KQL boundary')
        record('AirportOpsDataAgent source boundary', 'DataAgent', 'SUCCEEDED', 'Published exactly one Warehouse and one KQLDatabase source', data_agent_id, request_id(response))
    except Exception as exc:
        record('AirportOpsDataAgent source boundary', 'DataAgent', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
        if strict_mode:
            raise

In [ ]:
manifest_path = ROOT / 'deployment' / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else None
if manifest is None:
    record(str(manifest_path), 'DeploymentManifest', 'SKIPPED_PREREQUISITE', 'Mounted repository manifest was not found')
    if strict_mode and not dry_run:
        raise FileNotFoundError(str(manifest_path))


def split_sql_batches(script_text):
    return [batch.strip() for batch in re.split(r'^\s*GO\s*$', script_text, flags=re.MULTILINE | re.IGNORECASE) if batch.strip()]


if execute_warehouse_sql and manifest:
    if not warehouse_sql_endpoint and not dry_run:
        record('Warehouse SQL scripts', 'WarehouseSQL', 'SKIPPED_PREREQUISITE', 'warehouse_sql_endpoint runtime parameter is required')
        if strict_mode:
            raise ValueError('warehouse_sql_endpoint is required')
    elif dry_run:
        for relative_path in manifest['warehouse_scripts']:
            script_path = ROOT / relative_path
            status = 'DRY_RUN' if script_path.exists() else 'SKIPPED_PREREQUISITE'
            record(relative_path, 'WarehouseSQL', status, 'Would execute idempotent SQL script' if script_path.exists() else 'Script not found')
    else:
        import pyodbc

        access_token = notebookutils.credentials.getToken('https://database.windows.net/')
        token_bytes = access_token.encode('utf-16-le')
        token_struct = struct.pack('<I', len(token_bytes)) + token_bytes
        connection = pyodbc.connect(
            'Driver={ODBC Driver 18 for SQL Server};Server=' + warehouse_sql_endpoint +
            ';Database=' + warehouse_database_name + ';Encrypt=yes;TrustServerCertificate=no;',
            attrs_before={1256: token_struct}, autocommit=False,
        )
        try:
            for relative_path in manifest['warehouse_scripts']:
                script_path = ROOT / relative_path
                if not script_path.exists():
                    record(relative_path, 'WarehouseSQL', 'SKIPPED_PREREQUISITE', 'Script not found')
                    if strict_mode:
                        raise FileNotFoundError(str(script_path))
                    continue
                try:
                    cursor = connection.cursor()
                    for batch in split_sql_batches(script_path.read_text(encoding='utf-8')):
                        cursor.execute(batch)
                        while cursor.nextset():
                            pass
                    connection.commit()
                    record(relative_path, 'WarehouseSQL', 'SUCCEEDED', 'Executed all SQL batches')
                except Exception as exc:
                    connection.rollback()
                    record(relative_path, 'WarehouseSQL', 'FAILED', str(exc))
                    if strict_mode:
                        raise
        finally:
            connection.close()

In [ ]:
def kql_units(script_text):
    lines = [line for line in script_text.splitlines() if not line.lstrip().startswith('//')]
    if any(line.startswith('.') for line in lines):
        units = []
        current = []
        for line in lines:
            if line.startswith('.') and current:
                units.append('\n'.join(current).strip())
                current = []
            if line.strip() or current:
                current.append(line)
        if current:
            units.append('\n'.join(current).strip())
        return [unit for unit in units if unit]
    return [unit.strip() for unit in re.split(r'\n\s*\n', '\n'.join(lines)) if unit.strip()]


def execute_kql(command, management):
    endpoint = kql_query_uri.rstrip('/') + ('/v1/rest/mgmt' if management else '/v2/rest/query')
    response = requests.post(
        endpoint,
        headers=auth_headers(kql_token_audience),
        json={'db': kql_database_name, 'csl': command, 'properties': {'Options': {'queryconsistency': 'strongconsistency'}}},
        timeout=120,
    )
    if response.status_code != 200:
        raise FabricApiError(response.status_code, response.text[:4000], request_id(response))
    return response


if execute_eventhouse_kql and manifest:
    if not kql_query_uri and not dry_run:
        record('Eventhouse KQL scripts', 'EventhouseKQL', 'SKIPPED_PREREQUISITE', 'kql_query_uri runtime parameter is required')
        if strict_mode:
            raise ValueError('kql_query_uri is required')
    else:
        for relative_path in manifest['eventhouse_scripts']:
            script_path = ROOT / relative_path
            if not script_path.exists():
                record(relative_path, 'EventhouseKQL', 'SKIPPED_PREREQUISITE', 'Script not found')
                if strict_mode and not dry_run:
                    raise FileNotFoundError(str(script_path))
                continue
            units = kql_units(script_path.read_text(encoding='utf-8'))
            if dry_run:
                record(relative_path, 'EventhouseKQL', 'DRY_RUN', 'Would execute ' + str(len(units)) + ' KQL units')
                continue
            try:
                request_ids = []
                for unit in units:
                    response = execute_kql(unit, management=unit.lstrip().startswith('.'))
                    request_ids.append(request_id(response))
                record(relative_path, 'EventhouseKQL', 'SUCCEEDED', 'Executed ' + str(len(units)) + ' KQL units', request_id=';'.join(filter(None, request_ids)))
            except Exception as exc:
                record(relative_path, 'EventhouseKQL', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
                if strict_mode:
                    raise

In [ ]:
if deploy_bi_definitions:
    if not warehouse_sql_endpoint and not dry_run:
        record('AirportOpsSharedModel', 'SemanticModel', 'SKIPPED_PREREQUISITE', 'warehouse_sql_endpoint is required to replace TMDL parameters')
        if strict_mode:
            raise ValueError('warehouse_sql_endpoint is required for semantic model deployment')
    else:
        replacements = {
            '${WAREHOUSE_SERVER}': warehouse_sql_endpoint or '<RUNTIME_WAREHOUSE_SERVER>',
            '${WAREHOUSE_DATABASE}': warehouse_database_name,
        }
        semantic_model = deploy_definition(
            'AirportOpsSharedModel', 'SemanticModel',
            ROOT / 'semantic-model' / 'AirportOpsSharedModel.SemanticModel', replacements,
        )
        if semantic_model is not None:
            report_parts = report_definition_parts(
                ROOT / 'reports' / 'AirportOpsPersonaReports.Report', semantic_model['id'],
            )
            deploy_definition(
                'AirportOpsPersonaReports', 'Report',
                ROOT / 'reports' / 'AirportOpsPersonaReports.Report',
                parts_override=report_parts,
            )

if deploy_conditional_artifacts:
    deploy_data_agent()
    deploy_definition(
        'Airport Operations Command Center', 'FabricApp',
        ROOT / 'fabric-app', conditional=True,
    )
else:
    record('AirportOpsDataAgent', 'DataAgent', 'SKIPPED_PREREQUISITE', 'Set deploy_conditional_artifacts=true to configure and publish the native Data Agent')
    record('Airport Operations Command Center', 'FabricApp', 'SKIPPED_PREREQUISITE', 'Set deploy_conditional_artifacts=true to capability-check the target tenant')

record('Rayfin', 'ConfigurableAppModule', 'SKIPPED_UNSUPPORTED', 'No verified native Rayfin Fabric item/API; source module remains deployable through the app project')

if RESULTS:
    try:
        spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('deployment_results')
    except Exception as exc:
        print('SKIPPED_PREREQUISITE deployment_results Delta log:', str(exc))
    try:
        runtime_manifest = {'deployment_run_id': DEPLOYMENT_RUN_ID, 'environment_name': environment_name, 'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'git_commit': git_commit or None, 'artifacts': RESULTS}
        notebookutils.fs.put(deployment_manifest_output, json.dumps(runtime_manifest, default=str, indent=2), True)
        print('Published deployment manifest:', deployment_manifest_output)
    except Exception as exc:
        print('SKIPPED_PREREQUISITE JSON deployment manifest:', str(exc))

failed = [result for result in RESULTS if result['deployment_status'] == 'FAILED']
assert not failed, 'Platform deployment had ' + str(len(failed)) + ' failed artifacts'
print('Platform deployment complete:', len(RESULTS), 'recorded operations; dry_run=', dry_run)